# 04_ragas_evaluate

04_ragas_evaluate.py — Ragas evaluate() 로 여러 샘플·여러 메트릭 배치 평가

실무 표준 형태: EvaluationDataset + 메트릭 리스트 + evaluate() → DataFrame.

In [1]:
import os, sys, ssl, certifi
# Windows 인증서 저장소 손상 우회(임베딩/HTTPS 로드 SSL 에러 방지)
ssl.SSLContext.load_default_certs = lambda self, *a, **k: self.load_verify_locations(certifi.where())
# 노트북 커널엔 __file__ 이 없으므로 스크립트 호환 위해 정의 + supp/ 를 import 경로에 추가
__file__ = os.path.join(os.getcwd(), '04_ragas_evaluate.py')
sys.path.insert(0, os.path.abspath('..'))

In [2]:
"""
04_ragas_evaluate.py — Ragas evaluate() 로 여러 샘플·여러 메트릭 배치 평가

실무 표준 형태: EvaluationDataset + 메트릭 리스트 + evaluate() → DataFrame.
"""
import sys as _sys
from pathlib import Path as _Path
_sys.path.insert(0, str(_Path(__file__).resolve().parent.parent))

from ragas import SingleTurnSample, EvaluationDataset, evaluate
from ragas.metrics import (
    Faithfulness,
    ResponseRelevancy,
    LLMContextPrecisionWithReference,
    LLMContextRecall,
)

from _common import banner, llm_unavailable
from _judges import ragas_judge, ragas_embeddings


SAMPLES = [
    SingleTurnSample(
        user_input="에펠탑은 어디에 있나?",
        response="에펠탑은 프랑스 파리에 있습니다.",
        retrieved_contexts=["에펠탑은 프랑스 파리에 위치한다."],
        reference="에펠탑은 파리에 있다.",
    ),
    SingleTurnSample(
        user_input="에펠탑은 언제 지어졌나?",
        response="에펠탑은 1889년에 지어졌습니다.",
        retrieved_contexts=["에펠탑은 1889년 파리 만국박람회를 위해 세워졌다."],
        reference="에펠탑은 1889년에 지어졌다.",
    ),
    SingleTurnSample(
        user_input="에펠탑의 높이는?",
        response="에펠탑은 약 1000m 높이입니다.",   # 환각
        retrieved_contexts=["에펠탑의 높이는 약 330m 이다."],
        reference="약 330m.",
    ),
]


def main() -> None:
    banner("Ragas evaluate() — 3 샘플 × 4 메트릭 배치 평가")
    judge = ragas_judge()
    if judge is None:
        llm_unavailable()
        return

    dataset = EvaluationDataset(samples=SAMPLES)

    metrics = [
        Faithfulness(llm=judge),
        ResponseRelevancy(llm=judge, embeddings=ragas_embeddings()),
        LLMContextPrecisionWithReference(llm=judge),
        LLMContextRecall(llm=judge),
    ]

    result = evaluate(dataset=dataset, metrics=metrics)

    print("\n📊 메트릭별 평균 점수")
    print(f"  {result}")

    print("\n📋 샘플별 상세 (DataFrame)")
    df = result.to_pandas()
    cols = [c for c in df.columns if c in
            ("user_input", "faithfulness", "answer_relevancy",
             "llm_context_precision_with_reference", "context_recall")]
    print(df[cols].to_string(index=False))


if __name__ == "__main__":
    main()

D:\git\2604_agent_210h_handson\supp\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


C:\Users\user\AppData\Local\Temp\ipykernel_34468\3704471256.py:11: DeprecationWarning: Importing Faithfulness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import Faithfulness
  from ragas.metrics import (
C:\Users\user\AppData\Local\Temp\ipykernel_34468\3704471256.py:11: DeprecationWarning: Importing ResponseRelevancy from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import ResponseRelevancy
  from ragas.metrics import (
C:\Users\user\AppData\Local\Temp\ipykernel_34468\3704471256.py:11: DeprecationWarning: Importing LLMContextPrecisionWithReference from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import LLMContextPrecisionWithReference
  from ragas.metrics import (
C:\Users\user\


📌 Ragas evaluate() — 3 샘플 × 4 메트릭 배치 평가


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 6790.88it/s]

...

Evaluating:  83%|████████▎ | 10/12 [00:04<00:00,  2.16it/s]

Evaluating:  92%|█████████▏| 11/12 [00:05<00:00,  1.78it/s]

Evaluating: 100%|██████████| 12/12 [00:07<00:00,  1.36it/s]

Evaluating: 100%|██████████| 12/12 [00:07<00:00,  1.71it/s]


📊 메트릭별 평균 점수
  {'faithfulness': 0.6667, 'answer_relevancy': 0.8916, 'llm_context_precision_with_reference': 1.0000, 'context_recall': 1.0000}

📋 샘플별 상세 (DataFrame)
   user_input  faithfulness  answer_relevancy  llm_context_precision_with_reference  context_recall
 에펠탑은 어디에 있나?           1.0          0.994365                                   1.0             1.0
에펠탑은 언제 지어졌나?           1.0          0.821184                                   1.0             1.0
    에펠탑의 높이는?           0.0          0.859373                                   1.0             1.0
